# Módulo 5 — Del ranking a la operación: umbral, dinero y envejecimiento

Los módulos 2 a 4 dejan 13 detectores y un ranking por PR-AUC. Nada de eso es desplegable: en producción no llega un conjunto de prueba para ordenar, llega una transacción y hay que decir *sí* o *no*. Este módulo cubre lo que falta entre "tengo un score" y "tengo un sistema".

Y empieza corrigiendo una debilidad metodológica de los módulos anteriores: **el split era aleatorio sobre datos que tienen orden cronológico**. Eso mete transacciones del día 30 en el entrenamiento y del día 1 en la prueba — el modelo se evalúa sobre un pasado que ya vio.

Tres cambios respecto de los módulos anteriores, los tres deliberados:

| | Módulos 2-4 | Módulo 5 |
|---|---|---|
| Split | aleatorio | **temporal**: entrena con el pasado, evalúa con el futuro |
| Prevalencia en prueba | 14,1% (enriquecida) | **0,23%** (la real del período) |
| Métrica | PR-AUC, Precision@k | umbral, alertas por día, **pesos** |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.operations.costs import amount_concentration, break_even_review_cost
from src.operations.run_operations import (
    REVIEW_COST,
    evaluate_operationally,
    plot_calibration,
    plot_drift,
    plot_savings,
)
from src.operations.temporal import evaluate_by_period, get_temporal_data, volume_drift
from src.operations.thresholds import threshold_report
from src.unsupervised.benchmark import MODEL_LABELS, run_detectors

## 1. El split temporal

El corte cae en el step que deja el 70% de las **horas** por detrás — no el 70% de las filas. La distinción importa porque el volumen por hora no es constante en PaySim, y esa variación es justamente lo que interesa observar.

Se reservan además normales del período temprano que el detector **no** usa para ajustarse. Sirven para calibrar el umbral sin mirar los datos de ajuste, y en la sección 5 cumplen un segundo papel: permiten descartar una explicación.

In [ ]:
data = get_temporal_data()
X_train, X_test = data["X_train"], data["X_test"]
y_test, amounts, steps = data["y_test"], data["amounts_test"], data["steps_test"]

print(f"Corte en step={data['cutoff_step']}")
print(f"Train (normales del pasado):  {X_train.shape}")
print(f"Calib (normales retenidas):   {data['X_calib'].shape}")
print(f"Test (futuro, sin enriquecer): {X_test.shape} - fraude {int(y_test.sum())} ({y_test.mean():.4%})")

## 2. Por qué las métricas por conteo no alcanzan

El fraude de PaySim no solo es raro: es **caro**. Si un décimo de los casos explica la mitad del dinero, el orden que importa no es el de "más anómalo" sino el de "más caro entre los anómalos", y un detector puede ganar en PR-AUC mientras pierde plata.

In [ ]:
concentracion = amount_concentration(y_test, amounts)
equilibrio = break_even_review_cost(y_test, amounts)

print(f"monto mediano del fraude: {concentracion['median_fraud_amount']:,.0f}")
print(f"el 10% mas caro concentra: {concentracion['share_of_amount']:.1%} del monto defraudado")
print()
print(f"costo de revision de equilibrio: {equilibrio:,.0f} por alerta")
print(f"costo usado en el analisis:      {REVIEW_COST:,.0f} ({REVIEW_COST / equilibrio:.1f}x el equilibrio)")

Ese costo de equilibrio es la pérdida esperada por transacción. **Por debajo de él, el óptimo económico degenera en "revisar absolutamente todo"** y el umbral deja de ser una decisión de modelado: pasa a estar limitado por la capacidad del equipo, no por la economía. Conviene calcularlo *antes* de interpretar cualquier óptimo de ahorro, o se termina reportando como hallazgo algo que solo refleja el supuesto de costo.

In [ ]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_calib_scaled = scaler.transform(data["X_calib"])

results = run_detectors(X_train_scaled, X_test_scaled)
train_scores = {name: out["score_fn"](X_train_scaled) for name, out in results.items()}
calib_scores = {name: out["score_fn"](X_calib_scaled) for name, out in results.items()}

n_days = int(np.ceil((steps.max() - data["cutoff_step"]) / 24))
tabla = evaluate_operationally(results, y_test, amounts, calib_scores, n_days)
tabla.drop(columns=["clave"])

## 3. Comparar contra el Módulo 3 sin hacer trampa

Es tentador poner el PR-AUC de esta tabla al lado del que dio el Módulo 3 y anunciar un derrumbe. Sería incorrecto: **PR-AUC depende de la prevalencia**, y acá pasó del 14,1% al 0,23%. Buena parte de cualquier caída sería puramente mecánica.

Lo que sí es comparable es el **ROC-AUC**, que no depende de la prevalencia — y, sobre todo, el **orden** de los detectores.

In [ ]:
modulo3 = {
    "Gaussian Mixture": 0.948, "Local Outlier Factor": 0.932, "Deep SVDD": 0.860,
    "Mahalanobis robusto (MCD)": 0.896, "Autoencoder (ReLU)": 0.850,
    "kNN (k-esima distancia)": 0.853, "Isolation Forest": 0.850,
    "One-Class SVM (Nystrom)": 0.830, "PCA (reconstruccion)": 0.822,
    "HBOS": 0.800, "VAE (ELBO)": 0.731, "ECOD": 0.672, "MAD-z (baseline)": 0.383,
}

comparacion = tabla[["modelo", "roc_auc"]].copy()
comparacion["roc_auc_m3"] = comparacion["modelo"].map(modulo3)
comparacion["delta"] = comparacion["roc_auc"] - comparacion["roc_auc_m3"]
comparacion.sort_values("delta")

## 4. ¿El que mejor rankea es el que más plata salva?

No tiene por qué serlo. `recall_capacidad` cuenta fraudes; `recall_monto_capacidad` cuenta pesos. Ambos se miden en el mismo punto de operación: el umbral que produce las alertas que el equipo alcanza a revisar.

In [ ]:
mejor_ranking = tabla.iloc[0]
mejor_dinero = tabla.loc[tabla["ahorro_optimo"].idxmax()]

print(f"mejor por PR-AUC:      {mejor_ranking['modelo']} ({mejor_ranking['pr_auc']:.4f})")
print(f"mejor por ahorro neto: {mejor_dinero['modelo']} ({mejor_dinero['ahorro_optimo']:,.0f})")
print()
print(f"coinciden: {mejor_ranking['clave'] == mejor_dinero['clave']}")

In [ ]:
top = tabla["clave"].head(4).tolist()

fig = plot_savings(results, y_test, amounts, top, output_path=None)
plt.show()

## 5. El umbral sin etiquetas: sobre qué datos se calcula el cuantil

`quantile_threshold` es la única regla aplicable el día que se despliega el sistema: como el ajuste usa solo transacciones normales, el cuantil (1-α) debería dejar fuera una fracción α del tráfico legítimo. Es decir, **α es la tasa de falsos positivos prometida**.

Pero hay una decisión escondida ahí: ¿sobre qué datos se calcula el cuantil? Lo natural sería usar el propio conjunto de entrenamiento. Abajo se calculan las dos versiones —desde el entrenamiento y desde las normales retenidas— para ver cuánto importa.

In [ ]:
reportes = {n: threshold_report(calib_scores[n], results[n]["scores"], y_test) for n in top}
reportes_train = {n: threshold_report(train_scores[n], results[n]["scores"], y_test) for n in top}

for name in top:
    print(MODEL_LABELS.get(name, name))
    print(pd.DataFrame({
        "alpha": reportes[name]["alpha"],
        "obs_desde_train": reportes_train[name]["alpha_observado"],
        "obs_desde_calib": reportes[name]["alpha_observado"],
        "alertas": reportes[name]["alertas"],
        "recall": reportes[name]["recall"],
    }).to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    print()

Deep SVDD promete 0,1% de falsos positivos y entrega **35%**: tres órdenes de magnitud. Los otros tres detectores caen sobre la diagonal.

La hipótesis natural es el sobreajuste: Deep SVDD *minimiza explícitamente* la distancia al centro sobre los puntos de entrenamiento, así que sus scores ahí serían optimistas por construcción y el cuantil saldría demasiado bajo. Si fuera eso, calibrar sobre normales retenidas lo arreglaría.

**No lo arregla.** Las dos columnas de arriba son casi idénticas (0,389 contra 0,354). Y esa falta de diferencia es el dato: como ambos conjuntos vienen del período temprano, descarta el sobreajuste y deja una sola explicación en pie — la **escala del score se desplaza entre el período de ajuste y el de evaluación**.

Lo que hace el caso interesante es que ese mismo detector tiene el ranking más estable de los cuatro a lo largo del mes (sección 6). Estabilidad del orden y estabilidad de la escala son propiedades distintas: se puede tener la primera sin la segunda, y entonces ningún umbral fijo aprendido del pasado sirve. El arreglo no es calibrar mejor sobre datos viejos, es **recalibrar periódicamente sobre tráfico reciente**.

In [ ]:
fig = plot_calibration(reportes, reportes_train, output_path=None)
plt.show()

## 6. Envejecimiento: la métrica correcta importa

Acá hay una trampa que conviene mirar de frente. La primera versión de este análisis usaba PR-AUC por día y mostraba una **mejora espectacular** hacia el final del período: PR-AUC de 1.0 para los cuatro detectores el último día.

Era un artefacto. El volumen diario de PaySim se derrumba hacia el final del mes y el último día tiene 23 transacciones. Con esa muestra y la prevalencia disparada, cualquier detector saca métricas perfectas.

Dos correcciones, ambas necesarias:

- **filtrar los períodos con poco volumen** (`min_samples`), porque sobre 23 filas no se mide nada;
- **usar ROC-AUC en vez de PR-AUC**, porque la prevalencia diaria varía y el PR-AUC la sigue: una curva de PR-AUC por día mide el cambio de prevalencia, no la degradación del detector.

In [ ]:
volumen = volume_drift(steps, data["cutoff_step"])
print(f"volumen diario: primer dia={volumen['n'].iloc[0]:,}, ultimo dia={volumen['n'].iloc[-1]:,}")
print(f"razon: {volumen['n'].iloc[0] / max(1, volumen['n'].iloc[-1]):,.0f}x")

degradacion = {n: evaluate_by_period(y_test, results[n]["scores"], steps,
                                     data["cutoff_step"], roc_auc_score)
               for n in top}

print(f"dias que superan el filtro de volumen: {len(degradacion[top[0]])} de {len(volumen)}")

In [ ]:
fig = plot_drift(degradacion, volumen, output_path=None)
plt.show()

## 7. Conclusiones

- **Rankear bien y calibrar bien son dos propiedades distintas.** Un detector puede liderar el ranking y a la vez tener un umbral inservible. Elegir modelo por PR-AUC y dar el umbral por sentado es el error que este módulo hace visible.
- **Calibrar sobre datos retenidos es correcto, pero no siempre suficiente.** Acá se probó como arreglo del umbral de Deep SVDD y no cambió nada — lo que sirvió para descartar el sobreajuste y localizar la causa en la deriva temporal de la escala. Un arreglo que no funciona, medido, vale más que uno que se supone.
- **El PR-AUC del Módulo 3 no era comparable con este.** No por el split temporal en sí, sino porque cambió la prevalencia. La comparación honesta usa ROC-AUC, o directamente el orden de los detectores.
- **El dinero y el conteo no ordenan igual.** Con el 10% de los fraudes concentrando la mitad del monto, el detector que atrapa más casos no es necesariamente el que salva más plata.
- **Antes de optimizar un umbral, calcular el costo de equilibrio.** Si revisar cuesta menos que la pérdida esperada por transacción, el óptimo es "revisar todo" y no hay nada que optimizar: el problema es de capacidad, no de modelado.
- **Un artefacto de medición se parece mucho a un buen resultado.** El PR-AUC subiendo a 1.0 sobre 23 transacciones era la mejor noticia de la primera corrida y era falsa.